# Atoms of knowledge + licensing (replayable)

Every tracked file is an *atom* with a claim (OWN / OWN+TERMS / THIRD), a licence in context, a status and a hash.
Leaves roll up to a Merkle root per module, modules and external atoms to one AoK root. Same tree, same root.
This notebook rebuilds the index **in memory** (it never writes `kg/aok.json`), checks it, and shows where it differs
from the committed copy. **Swap data:** point `REPO` at any git checkout (yours, a fork, a BYO build) and `AOK_JSON` at
the index you want to check it against.

In [1]:
import os, sys, json
from pathlib import Path
if Path.cwd().name == 'notebooks': os.chdir('..')
sys.path.insert(0, os.getcwd())
from collections import Counter
import pandas as pd
from baebugs import aok, compare
from baebugs.explore import merkle_root

REPO = '.'                 # swap: any git checkout
AOK_JSON = 'kg/aok.json'   # swap: the index to check against (None to skip)

In [2]:
A = aok.build(REPO)
errs = aok.check(A, REPO)
print('AoK root', A['root'], '|', A['atoms'], 'atoms |', A['totals'], '| check errors:', len(errs))
assert aok.build(REPO)['root'] == A['root'], 'not deterministic'
errs[:5]

AoK root e42b59c5456308c0a12a47d67ecc8a31e4c5d33cf19c5d41d1d57ff8ac299828 | 761 atoms | {'OWN': 696, 'OWN+TERMS': 45, 'THIRD': 20} | check errors: 0


[]

## Modules: claims, outside terms, roots

In [3]:
pd.DataFrame([{'module': m, 'atoms': d['atoms'], **d['claims'], 'terms': ', '.join(t.split(':')[1] for t in d['terms']) or '-',
               'root': d['root'][:12]} for m, d in A['modules'].items()]).fillna(0)

,module,atoms,OWN,terms,root,OWN+TERMS
0,custody-ledgers,33,33.0,-,8f48e4f879b1,0.0
1,daisy-protocols,17,17.0,-,401e52c7913d,0.0
2,daisy-results,11,0.0,"gbif-media, ultralytics",b071c5cb0af3,11.0
3,dashboards,16,16.0,-,350619827ff6,0.0
4,docs,24,24.0,-,6b4f845f2ad1,0.0
5,kaggle-kernels,10,0.0,"gbif-media, ultralytics",ff1a3e54c36a,10.0
6,knowledge-graph,5,0.0,gbif-media,f76004ad5e8e,5.0
7,lab-parts,106,106.0,-,a974e369a4d0,0.0
8,mvp-site,30,30.0,-,e9559d478120,0.0
9,notebooks,21,21.0,-,f88458c36b4b,0.0


## Not ours: external atoms (credited, pinned, never claimed)

In [4]:
pd.DataFrame([{'id': e['id'], 'licence': e['licence'], 'pin': (e.get('pin') or {}).get('commit', '')[:10], 'effect': e['effect'][:90]}
              for e in A['external']])

,id,licence,pin,effect
0,ext:ultralytics,AGPL-3.0,,models trained or run with it are AGPL-3.0 unl...
1,ext:gbif-media,CC0-1.0 / CC-BY-4.0,,CC BY images need the rights holder credited i...
2,ext:opentrons,Apache-2.0,,our labware and protocols are our own; 'Opentr...
3,ext:threejs,MIT,,none beyond keeping its notice
4,ext:manifold3d,Apache-2.0,,none on the generated meshes
5,ext:trimesh,MIT,,none on the generated meshes
6,ext:kaggle,Kaggle Terms of Use,,private datasets/kernels stay private; outputs...
7,ext:claude,Anthropic Terms (outputs assigned to the user),,"biobitworks directed, reviewed and chose the w..."
8,ext:upstream:BioKEA/BugPicker,GPL-3.0,1ef0b97e63,"pinned by commit + sha256 root only, never cop..."
9,ext:upstream:BioKEA/BugPicker-arm-demo,GPL-3.0,bfd3be6ac8,"pinned by commit + sha256 root only, never cop..."


## Recompute one atom and its module root by hand
A leaf hash covers path, bytes, claim, licence and status, so changing *any* of them (even only the licence) moves the
module root and the AoK root.

In [5]:
x = next(a for a in A['leaves'] if a['path'] == 'baebugs/train.py')
print(x['claim'], '|', x['licence'], '|', x['status'])
assert aok.leaf_hash(x) == x['leaf'] and aok._sha(Path(REPO, x['path'])) == x['sha256']
mod = [a for a in A['leaves'] if a['module'] == x['module']]
assert merkle_root(sorted(a['leaf'] for a in mod)) == A['modules'][x['module']]['root']
relicensed = {**x, 'licence': 'MIT'}       # pretend the AGPL term was dropped
print('leaf', x['leaf'][:12], '->', aok.leaf_hash(relicensed)[:12], '(licence alone moves the leaf)')

OWN+TERMS | MIT + ultralytics terms | built
leaf d4ad065a16a0 -> 6634355326f4 (licence alone moves the leaf)


## Is the committed index current?
Top-down compare of the in-memory build against the committed index: SAME modules are skipped, changed modules are
break points with the leaves that moved. On a dirty tree (or after adding these notebooks) you will see break points;
regenerate with `python -m baebugs.aok`.

In [6]:
if AOK_JSON and Path(REPO, AOK_JSON).exists():
    committed = json.loads(Path(REPO, AOK_JSON).read_text())
    r = compare.compare(committed, A, 'committed', 'tree')
    print('identical' if r['identical'] else f"break points: {r['break_points']}")
    for m in r['break_points']:
        d = r['modules'][m]
        print(' ', m, '| differs', d['DIFFERS'][:3], '| new', d['ONLY_tree'][:3], '| gone', d['ONLY_committed'][:3])

break points: ['notebooks']
  notebooks | differs ['notebooks/aok.ipynb', 'notebooks/compare_byo.ipynb', 'notebooks/mvp_graph_pack.ipynb'] | new [] | gone []


## The people version (`docs/CLAIMS.md`), rendered here without writing it

In [7]:
print('\n'.join(aok.claims_md(A).splitlines()[:12]))

# What baebugs claims, and under which licence

Every file is an *atom* with a claim, a licence in context, a status and a hash (full list for agents: [`kg/aok.json`](../kg/aok.json)). This page is the short version. It records what the licences say; it is not legal advice.

**AoK root:** `e42b59c5456308c0a12a47d67ecc8a31e4c5d33cf19c5d41d1d57ff8ac299828` · 761 atoms · OWN 696 · OWN+TERMS 45 · THIRD 20

| claim | meaning |
|---|---|
| **OWN** | written here; claimable; MIT |
| **OWN+TERMS** | written here and claimable; the listed outside terms also apply in this context |
| **THIRD** | not ours; used under its licence; credited, pinned by hash, never claimed |

